# Feature Stores — Offline, Online, Point-in-Time Join, Serving

## Mental Model

A feature store is the contract between **training time** and **serving time**.

It solves four hard production problems:

1. **Feature reuse** — teams define features once and reuse them.
2. **Training-serving skew** — the same feature logic is used both offline and online.
3. **Point-in-time correctness** — historical training rows only see data available at that moment.
4. **Operational serving** — low-latency lookup of fresh features for inference.

In this notebook we use a Citi-style telemetry domain with:

- **endpoints**: 10,000 rows
- **metrics**: 500,000 rows
- **alerts**: 25,000 rows
- **Narrative**: 6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers

We demonstrate:

- Feast repo setup
- entity + feature definition
- historical retrieval with point-in-time join
- online materialization into SQLite
- why naive joins leak future data
- architecture comparison across common feature store platforms

In [ ]:
import json
import os
import sqlite3
import tempfile
import time
from pathlib import Path

import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

import feast
from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.repo_config import RepoConfig
from feast.types import Float32, Int64, String

PROJECT_ID = "citi_features"
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!"
}

WORKDIR = Path("/mnt/data/feast_citi_repo")
WORKDIR.mkdir(parents=True, exist_ok=True)

print("Feast version:", feast.__version__)
print("Feature repo path:", WORKDIR)
print("Project ID:", PROJECT_ID)

In [ ]:
def get_pg_conn():
    return psycopg2.connect(**DB_CONFIG)

def query_df(sql: str, params=None) -> pd.DataFrame:
    with get_pg_conn() as conn:
        return pd.read_sql(sql, conn, params=params)

check_sql = '''
SELECT
    (SELECT COUNT(*) FROM endpoints) AS endpoints_count,
    (SELECT COUNT(*) FROM metrics) AS metrics_count,
    (SELECT COUNT(*) FROM alerts) AS alerts_count
'''
counts_df = query_df(check_sql)
display(counts_df)

In [ ]:
# Build a daily alert feature table in local files from PostgreSQL source data.
# This keeps the notebook executable and still demonstrates Feast concepts cleanly.

offline_feature_sql = '''
WITH alert_daily AS (
    SELECT
        a.endpoint_id,
        DATE_TRUNC('day', a.created_at) AS event_timestamp,
        COUNT(*)::int AS alert_count_24h,
        SUM(CASE WHEN LOWER(a.severity) IN ('critical', 'sev1') THEN 1 ELSE 0 END)::int AS critical_alert_count_24h,
        MAX(
            CASE LOWER(a.severity)
                WHEN 'critical' THEN 'critical'
                WHEN 'sev1' THEN 'critical'
                WHEN 'high' THEN 'high'
                WHEN 'sev2' THEN 'high'
                WHEN 'medium' THEN 'medium'
                WHEN 'low' THEN 'low'
                ELSE 'unknown'
            END
        ) AS last_severity,
        (
            COUNT(*)::float /
            NULLIF(COUNT(DISTINCT DATE_TRUNC('hour', a.created_at)), 0)
        )::float AS alert_rate
    FROM alerts a
    GROUP BY a.endpoint_id, DATE_TRUNC('day', a.created_at)
)
SELECT
    endpoint_id,
    event_timestamp,
    event_timestamp AS created_timestamp,
    alert_count_24h,
    critical_alert_count_24h,
    last_severity,
    alert_rate
FROM alert_daily
ORDER BY event_timestamp, endpoint_id
'''

features_df = query_df(offline_feature_sql)
features_path = WORKDIR / "endpoint_alert_features.parquet"
features_df.to_parquet(features_path, index=False)

print("Offline feature rows:", len(features_df))
print("Offline feature file:", features_path)
display(features_df.head())

In [ ]:
# Create an entity dataframe for historical retrieval: one row per training event.
training_entity_sql = '''
WITH latest_metrics AS (
    SELECT
        m.endpoint_id,
        DATE_TRUNC('day', m.timestamp) AS event_timestamp,
        MAX(CASE WHEN m.metric_name = 'latency' THEN m.value END) AS latency,
        MAX(CASE WHEN m.metric_name = 'error_rate' THEN m.value END) AS error_rate,
        MAX(CASE WHEN m.metric_name = 'throughput' THEN m.value END) AS throughput
    FROM metrics m
    GROUP BY m.endpoint_id, DATE_TRUNC('day', m.timestamp)
),
future_alerts AS (
    SELECT
        a.endpoint_id,
        DATE_TRUNC('day', a.created_at - INTERVAL '1 day') AS event_timestamp,
        COUNT(*)::int AS next_day_alerts
    FROM alerts a
    GROUP BY a.endpoint_id, DATE_TRUNC('day', a.created_at - INTERVAL '1 day')
)
SELECT
    lm.endpoint_id,
    lm.event_timestamp,
    COALESCE(lm.latency, 0.0) AS latency,
    COALESCE(lm.error_rate, 0.0) AS error_rate,
    COALESCE(lm.throughput, 0.0) AS throughput,
    CASE WHEN COALESCE(fa.next_day_alerts, 0) > 0 THEN 1 ELSE 0 END AS label_next_day_alert
FROM latest_metrics lm
LEFT JOIN future_alerts fa
  ON fa.endpoint_id = lm.endpoint_id
 AND fa.event_timestamp = lm.event_timestamp
ORDER BY lm.event_timestamp DESC, lm.endpoint_id
LIMIT 5000
'''
entity_df = query_df(training_entity_sql)
entity_path = WORKDIR / "training_entities.parquet"
entity_df.to_parquet(entity_path, index=False)

print("Entity dataframe rows:", len(entity_df))
display(entity_df.head())

In [ ]:
# Define Feast repo files.
feature_store_yaml = f'''
project: {PROJECT_ID}
registry: data/registry.db
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 2
'''

feature_repo_py = '''
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String

endpoint_entity = Entity(
    name="endpoint_id",
    join_keys=["endpoint_id"],
    value_type=Int64,
    description="Unique API endpoint identifier"
)

endpoint_alert_source = FileSource(
    name="endpoint_alert_source",
    path="endpoint_alert_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

endpoint_alert_features = FeatureView(
    name="endpoint_alert_features",
    entities=[endpoint_entity],
    ttl=None,
    schema=[
        Field(name="alert_count_24h", dtype=Int64),
        Field(name="critical_alert_count_24h", dtype=Int64),
        Field(name="last_severity", dtype=String),
        Field(name="alert_rate", dtype=Float32),
    ],
    online=True,
    source=endpoint_alert_source,
    tags={"domain": "citi-telemetry"}
)
'''

(WORKDIR / "feature_store.yaml").write_text(feature_store_yaml, encoding="utf-8")
(WORKDIR / "example_repo.py").write_text(feature_repo_py, encoding="utf-8")

data_dir = WORKDIR / "data"
data_dir.mkdir(exist_ok=True)

print((WORKDIR / "feature_store.yaml").read_text(encoding="utf-8"))
print("-----")
print((WORKDIR / "example_repo.py").read_text(encoding="utf-8")[:800])

In [ ]:
# Load the local repo and apply the feature definitions.
store = FeatureStore(repo_path=str(WORKDIR))
store.apply([store._registry.project_metadata])  # ensure registry init

# Import objects from repo file dynamically so the notebook remains self-contained.
import importlib.util
spec = importlib.util.spec_from_file_location("example_repo", WORKDIR / "example_repo.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

store.apply([
    module.endpoint_entity,
    module.endpoint_alert_features,
])

print("Feast apply completed.")
print("Registered entities:", [e.name for e in store.list_entities()])
print("Registered feature views:", [fv.name for fv in store.list_feature_views()])

In [ ]:
# Historical retrieval with point-in-time correctness.
historical_features = store.get_historical_features(
    entity_df=entity_df[["endpoint_id", "event_timestamp"]],
    features=[
        "endpoint_alert_features:alert_count_24h",
        "endpoint_alert_features:critical_alert_count_24h",
        "endpoint_alert_features:last_severity",
        "endpoint_alert_features:alert_rate",
    ],
).to_df()

training_dataset = entity_df.merge(
    historical_features,
    on=["endpoint_id", "event_timestamp"],
    how="left"
)

print(f"Training dataset: {training_dataset.shape[0]} rows × {training_dataset.shape[1]} features")
display(training_dataset.head())

In [ ]:
# Materialize latest features to the SQLite online store.
start_ts = pd.to_datetime(features_df["event_timestamp"]).min().to_pydatetime()
end_ts = pd.to_datetime(features_df["event_timestamp"]).max().to_pydatetime()

store.materialize(start_date=start_ts, end_date=end_ts)

sample_ids = (
    training_dataset["endpoint_id"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .head(10)
    .tolist()
)

feature_refs = [
    "endpoint_alert_features:alert_count_24h",
    "endpoint_alert_features:critical_alert_count_24h",
    "endpoint_alert_features:last_severity",
    "endpoint_alert_features:alert_rate",
]

t0 = time.perf_counter()
online_response = store.get_online_features(
    features=feature_refs,
    entity_rows=[{"endpoint_id": eid} for eid in sample_ids],
).to_dict()
elapsed_ms = (time.perf_counter() - t0) * 1000

online_df = pd.DataFrame(online_response)
print(f"Online feature retrieval: {elapsed_ms:.2f}ms")
display(online_df)

In [ ]:
# Point-in-time join deep dive:
# Correct join: use features available as of the row timestamp.
# Wrong join: attach future alerts from the following day, which leaks signal.

pit_df = training_dataset.copy()

wrong_join_sql = '''
WITH future_alert_signal AS (
    SELECT
        a.endpoint_id,
        DATE_TRUNC('day', a.created_at - INTERVAL '1 day') AS event_timestamp,
        COUNT(*)::int AS leaked_future_alerts
    FROM alerts a
    GROUP BY a.endpoint_id, DATE_TRUNC('day', a.created_at - INTERVAL '1 day')
)
SELECT
    endpoint_id,
    event_timestamp,
    leaked_future_alerts
FROM future_alert_signal
'''
wrong_df = query_df(wrong_join_sql)

compare_df = pit_df.merge(
    wrong_df,
    on=["endpoint_id", "event_timestamp"],
    how="left"
).fillna({"leaked_future_alerts": 0})

# Build a simple demonstration score:
# correct model proxy uses point-in-time-safe feature
# wrong model proxy uses leaked future signal
correct_pred = (compare_df["alert_count_24h"].fillna(0) > 0).astype(int)
wrong_pred = (compare_df["leaked_future_alerts"].fillna(0) > 0).astype(int)
label = compare_df["label_next_day_alert"].astype(int)

correct_acc = float((correct_pred == label).mean())
wrong_acc = float((wrong_pred == label).mean())

# Force the narrative to highlight a classic leakage pattern while staying data-driven.
display(compare_df[[
    "endpoint_id", "event_timestamp", "label_next_day_alert",
    "alert_count_24h", "leaked_future_alerts"
]].head(10))

print("Timestamp alignment logic:")
print("- Safe join: feature.event_timestamp <= training_row.event_timestamp")
print("- Wrong join: future alert events accidentally joined back to the prior training row")
print(f"Point-in-time-safe proxy accuracy: {correct_acc:.4f}")
print(f"Naive leaked-join proxy accuracy: {wrong_acc:.4f}")
print(f"Accuracy lift from leakage: {(wrong_acc - correct_acc) * 100:.2f}%")

In [ ]:
architecture_df = pd.DataFrame([
    {
        "platform": "Feast",
        "open_source": "Yes",
        "online_latency": "Low ms with Redis/SQLite/online plugins",
        "offline_batch": "Yes",
        "streaming_features": "Partial / ecosystem-driven",
        "cost": "Low to medium"
    },
    {
        "platform": "Tecton",
        "open_source": "No",
        "online_latency": "Low ms",
        "offline_batch": "Yes",
        "streaming_features": "Strong",
        "cost": "High"
    },
    {
        "platform": "Hopsworks",
        "open_source": "Yes",
        "online_latency": "Low ms",
        "offline_batch": "Yes",
        "streaming_features": "Strong",
        "cost": "Medium"
    },
    {
        "platform": "SageMaker Feature Store",
        "open_source": "No",
        "online_latency": "Low ms in AWS",
        "offline_batch": "Yes",
        "streaming_features": "AWS-native patterns",
        "cost": "Usage-based"
    }
])

display(architecture_df)

## What Just Happened

A feature store prevents the **#1 ML production bug: training-serving skew**.

The same feature definitions are used for:

- **training** via offline / historical retrieval
- **serving** via online lookup

And the critical guardrail is the **point-in-time join**:

- it ensures a training row only sees information known at that moment
- it prevents future data leakage
- it keeps offline evaluation honest

In a Citi-style risk environment, features used to predict an event must reflect what was known **before** that event. A common operating rule is to compute features from a stable cutoff, such as **24 hours before the target event**.